# Notebook 02 — Silver Transformation

**Project:** End-to-End Retail Data Engineering Pipeline (Olist)

## Objective
Turn Bronze into **typed, validated, de-duplicated Silver tables**, and keep every rejected record in a **quarantine table** with the reason.

## What happens to each Bronze row
1. **Type conversion** — strings become timestamps, integers, decimals or doubles. A non-empty value that cannot be converted is a failure, never silently turned into NULL.
2. **Business-rule validation** — each rule is plain business logic (no test-data markers). A row that breaks any rule goes to `quarantine_<table>` with its reasons.
3. **De-duplication** — among valid rows, one row per business key is kept: the one from the latest batch.
4. **Reconciliation** — `Bronze rows = Silver rows + quarantined rows + duplicates/superseded rows removed`, and Silver business keys must be unique.

## Business keys
| Table | Key |
|---|---|
| orders | `order_id` |
| customers | `customer_id` |
| order_items | `order_id`, `order_item_id` |
| order_payments | `order_id`, `payment_sequential` |
| order_reviews | `review_id`, `order_id` *(a review ID can legitimately appear on several orders)* |
| products / sellers | `product_id` / `seller_id` |
| category_translation | `product_category_name` |
| geolocation | all five columns (exact duplicate removal) |

Source: `bronze_*` tables. Destination: `silver_*` and `quarantine_*` tables. Run Notebook 01 first.


## 1. Configuration and Reusable Function

`build_silver()` applies the four steps above to any table, driven by its keys, casts and rules. Using one function keeps every table consistent and means the logic is written (and tested) once.

- `casts` — column → Spark type, using `try_cast` (returns NULL instead of failing on bad text).
- `rules` — rule name → SQL condition that must be **true** for a valid row (NULL counts as a failure unless the rule allows it explicitly).
- Quarantine rows keep the original record as JSON in `_raw_record`, so nothing is lost.
- Ties between duplicates are broken deterministically: latest `_batch_date`, then latest `_source_file`.

In [1]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SUMMARY = {}   # table -> (bronze, silver, quarantined, removed); filled by build_silver()


def present(col):
    return f"{col} IS NOT NULL AND length(trim({col})) > 0"


def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)


def build_silver(name, keys, rules, casts=None, renames=None):
    casts, renames = casts or {}, renames or {}
    bronze = spark.table(f"bronze_{name}")
    raw_cols = [c for c in bronze.columns if not c.startswith("_")]
    df = bronze.withColumn("_raw_record", F.to_json(F.struct(*raw_cols)))

    # 1. Type conversion. A non-null value that fails to convert is recorded as a failure.
    fail_cols = []
    for col, dtype in casts.items():
        typed = F.expr(f"try_cast(`{col}` AS {dtype})")
        df = (df.withColumn(f"_fail_{col}", F.when(F.col(col).isNotNull() & typed.isNull(), F.lit(f"unparseable_{col}")))
                .withColumn(col, typed))
        fail_cols.append(F.col(f"_fail_{col}"))
    for old, new in renames.items():
        df = df.withColumnRenamed(old, new)

    # 2. Business rules (NULL result = failure).
    rule_cols = [F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(rule)) for rule, cond in rules.items()]
    df = (df.withColumn("_dq_failures", F.filter(F.array(*fail_cols, *rule_cols), lambda x: x.isNotNull()))
            .drop(*[f"_fail_{c}" for c in casts]))
    invalid = df.filter(F.size("_dq_failures") > 0)
    valid = df.filter(F.size("_dq_failures") == 0)

    save(invalid.select(
        F.lit(name).alias("_table"), "_raw_record",
        F.concat_ws(";", "_dq_failures").alias("_dq_failures"),
        "_source_file", "_batch_date", F.current_timestamp().alias("_quarantined_at")),
        f"quarantine_{name}")

    # 3. De-duplicate: latest batch wins per business key.
    w = Window.partitionBy(*keys).orderBy(F.col("_batch_date").desc(), F.col("_source_file").desc())
    silver = (valid.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
                   .drop("_rn", "_raw_record", "_dq_failures"))
    save(silver, f"silver_{name}")

    # 4. Reconciliation.
    s = spark.table(f"silver_{name}")
    n_bronze, n_silver = bronze.count(), s.count()
    n_bad = spark.table(f"quarantine_{name}").count()
    duplicate_keys = n_silver - s.select(*keys).distinct().count()
    assert duplicate_keys == 0, f"{name}: Silver has {duplicate_keys} duplicate business keys"
    SUMMARY[name] = (n_bronze, n_silver, n_bad, n_bronze - n_silver - n_bad)
    print(f"silver_{name}: bronze={n_bronze:,} silver={n_silver:,} quarantined={n_bad:,} removed_duplicates={n_bronze - n_silver - n_bad:,}")

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 3, Finished, Available, Finished, False)

## 2. Silver Orders

Orders arrive more than once: an early *processing* snapshot, the final record in a later batch, and some re-delivered copies. Keeping the **latest batch** per `order_id` gives the current order status.

An order is invalid if it has no ID or customer, a status outside the allowed set, no valid purchase timestamp, or a delivery date earlier than the purchase date.


In [2]:
build_silver(
    "orders",
    keys=["order_id"],
    casts={c: "timestamp" for c in [
        "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date"]},
    rules={
        "order_id_present": present("order_id"),
        "customer_id_present": present("customer_id"),
        "order_status_valid": "order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled')",
        "purchase_timestamp_present": "order_purchase_timestamp IS NOT NULL",
        "delivery_not_before_purchase": "order_delivered_customer_date IS NULL OR order_delivered_customer_date >= order_purchase_timestamp",
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 4, Finished, Available, Finished, False)

silver_orders: bronze=136,570 silver=99,441 quarantined=20 removed_duplicates=37,109


## 3. Silver Customers

`customer_id` is the per-order customer key (the same person has a different `customer_id` on each order; `customer_unique_id` identifies the person). Re-delivered duplicate rows are removed.


In [3]:
build_silver(
    "customers",
    keys=["customer_id"],
    rules={
        "customer_id_present": present("customer_id"),
        "customer_unique_id_present": present("customer_unique_id"),
        "customer_state_two_letters": "customer_state IS NOT NULL AND length(trim(customer_state)) = 2",
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 5, Finished, Available, Finished, False)

silver_customers: bronze=99,938 silver=99,441 quarantined=0 removed_duplicates=497


## 4. Silver Order Items

One row per item in an order. Prices and freight are converted to `decimal(12,2)` and must not be negative.


In [4]:
build_silver(
    "order_items",
    keys=["order_id", "order_item_id"],
    casts={"order_item_id": "int", "shipping_limit_date": "timestamp",
           "price": "decimal(12,2)", "freight_value": "decimal(12,2)"},
    rules={
        "order_id_present": present("order_id"),
        "order_item_id_present": "order_item_id IS NOT NULL",
        "product_id_present": present("product_id"),
        "seller_id_present": present("seller_id"),
        "price_non_negative": "price IS NOT NULL AND price >= 0",
        "freight_non_negative": "freight_value IS NOT NULL AND freight_value >= 0",
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 6, Finished, Available, Finished, False)

silver_order_items: bronze=113,796 silver=112,650 quarantined=20 removed_duplicates=1,126


## 5. Silver Order Payments

One row per payment instalment of an order. Payment value must be non-negative and the payment type must be one of the known Olist types.



In [5]:
build_silver(
    "order_payments",
    keys=["order_id", "payment_sequential"],
    casts={"payment_sequential": "int", "payment_installments": "int", "payment_value": "decimal(12,2)"},
    rules={
        "order_id_present": present("order_id"),
        "payment_sequential_present": "payment_sequential IS NOT NULL",
        "payment_type_valid": "payment_type IN ('credit_card','boleto','voucher','debit_card','not_defined')",
        "payment_value_non_negative": "payment_value IS NOT NULL AND payment_value >= 0",
        "installments_non_negative": "payment_installments IS NOT NULL AND payment_installments >= 0",
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 7, Finished, Available, Finished, False)

silver_order_payments: bronze=104,945 silver=103,886 quarantined=20 removed_duplicates=1,039


## 6. Silver Order Reviews

The business key is **`(review_id, order_id)`**. In the Olist data one `review_id` can belong to several orders, so de-duplicating on `review_id` alone deletes legitimate reviews (this removed 814 rows earlier). Only exact re-deliveries of the same review on the same order are removed.

Review score must be between 1 and 5.

In [6]:
build_silver(
    "order_reviews",
    keys=["review_id", "order_id"],
    casts={"review_score": "int", "review_creation_date": "timestamp", "review_answer_timestamp": "timestamp"},
    rules={
        "review_id_present": present("review_id"),
        "order_id_present": present("order_id"),
        "review_score_between_1_and_5": "review_score BETWEEN 1 AND 5",
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 8, Finished, Available, Finished, False)

silver_order_reviews: bronze=100,236 silver=99,224 quarantined=20 removed_duplicates=992


## 7. Silver Products

Numeric attributes are converted to integers. The two misspelled source columns (`..._lenght`) are renamed. A missing category or measurement is allowed (it is real source data); a negative weight is not.


In [7]:
build_silver(
    "products",
    keys=["product_id"],
    casts={c: "int" for c in [
        "product_name_lenght", "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]},
    renames={"product_name_lenght": "product_name_length",
             "product_description_lenght": "product_description_length"},
    rules={
        "product_id_present": present("product_id"),
        "weight_non_negative": "product_weight_g IS NULL OR product_weight_g >= 0",
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 9, Finished, Available, Finished, False)

silver_products: bronze=32,951 silver=32,951 quarantined=0 removed_duplicates=0


## 8. Silver Sellers



In [8]:
build_silver(
    "sellers",
    keys=["seller_id"],
    rules={"seller_id_present": present("seller_id")},
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 10, Finished, Available, Finished, False)

silver_sellers: bronze=3,095 silver=3,095 quarantined=0 removed_duplicates=0


## 9. Silver Category Translation

Maps Portuguese product categories to English.



In [9]:
build_silver(
    "category_translation",
    keys=["product_category_name"],
    rules={
        "category_present": present("product_category_name"),
        "english_name_present": present("product_category_name_english"),
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 11, Finished, Available, Finished, False)

silver_category_translation: bronze=71 silver=71 quarantined=0 removed_duplicates=0


## 10. Silver Geolocation

Coordinates are converted to doubles and must be valid latitude/longitude values. The source has many repeated rows for the same postal-code prefix, so exact duplicates (all five columns) are removed.

In [10]:
build_silver(
    "geolocation",
    keys=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"],
    casts={"geolocation_lat": "double", "geolocation_lng": "double"},
    rules={
        "zip_prefix_present": present("geolocation_zip_code_prefix"),
        "latitude_valid": "geolocation_lat BETWEEN -90 AND 90",
        "longitude_valid": "geolocation_lng BETWEEN -180 AND 180",
    },
)

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 12, Finished, Available, Finished, False)

silver_geolocation: bronze=1,000,163 silver=738,332 quarantined=0 removed_duplicates=261,831


## 11. Reconcile Silver With the Source Manifest

Compare each Silver table with `expected_silver_rows` and each quarantine table with `injected_bad_rows` from the manifest (both are computed by the source-simulation script from the original data, independent of this notebook).

Expected output: every `silver_vs_manifest` and `quarantine_vs_injected` value is 0, and `bronze = silver + quarantined + removed` for every table.

If a value is **not** 0, do not adjust the rules to force a match. Inspect the rows (a quarantine count above the injected count would mean the real source data contains additional rule violations).

In [11]:
manifest = (
    spark.read.option("multiLine", "true").json("Files/retail/source/manifest.json")
    .first().asDict(recursive=True)["tables"]
)

rows = [(t, b, s, q, r, manifest[t]["expected_silver_rows"], manifest[t].get("injected_bad_rows") or 0)
        for t, (b, s, q, r) in SUMMARY.items()]

report = (
    spark.createDataFrame(rows, ["table_name", "bronze_rows", "silver_rows", "quarantined_rows",
                                 "duplicates_removed", "manifest_expected_silver", "manifest_injected_bad"])
    .withColumn("silver_vs_manifest", F.col("silver_rows") - F.col("manifest_expected_silver"))
    .withColumn("quarantine_vs_injected", F.col("quarantined_rows") - F.col("manifest_injected_bad"))
)
display(report.orderBy("table_name"))

assert len(SUMMARY) == 9, "Run every Silver section above before reconciling."
assert report.filter("silver_vs_manifest != 0 OR quarantine_vs_injected != 0").count() == 0, \
    "Silver does not match the manifest - inspect the differing tables; do not force the counts."
print("Silver layer validated: all 9 tables reconcile with the manifest.")

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cdc70f27-e2b6-4009-9849-2324981e341c)

Silver layer validated: all 9 tables reconcile with the manifest.


## 12. Quarantine Summary

Show which rule rejected how many records. This is the evidence that bad records are caught by business rules and kept for investigation instead of being discarded.

In [12]:
quarantine = reduce(
    lambda a, b: a.unionByName(b),
    [spark.table(f"quarantine_{t}").select("_table", "_dq_failures") for t in SUMMARY],
)
display(quarantine.groupBy("_table", "_dq_failures").count().orderBy("_table", "_dq_failures"))

StatementMeta(, 2f711ec3-b3dd-4289-bb98-538386855432, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 40f61449-a412-4402-8f1c-0a576a1352f0)

## 13. Summary

| Item | Result |
|---|---|
| Silver tables | 9 typed, validated, de-duplicated Delta tables (`silver_*`) |
| Quarantine | One table per source table (`quarantine_*`) with reasons and the original record |
| Keys | Verified unique in every Silver table |
| Reconciliation | `Bronze = Silver + quarantined + duplicates removed`; Silver and quarantine counts match the manifest |

**Limits to state honestly:** Silver is rebuilt in full on every run (no incremental/MERGE), and the source data is a simulated delivery of a static public dataset.

Next: Notebook 03 builds the Gold layer from these Silver tables.